# Results Analysis: PU Learning Experiments

This notebook provides interactive analysis of PN, uPU, and nnPU experiment results using hvPlot + HoloViews + Bokeh.

**Prerequisites**: Run experiments first to generate results:
```bash
# Quick test (10 epochs, 2 seeds)
python experiments/scripts/run_all.py --seeds 42 43 --quick

# Full experiments (100 epochs, 5 seeds)
python experiments/scripts/run_all.py --seeds 42 43 44 45 46
```

In [ ]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa: F401
import pandas as pd

from pu_learning.utils.visualization import (
    create_comparison_dashboard,
    load_experiment_results,
    plot_final_performance,
    plot_training_curves,
    print_summary_statistics,
)

# Enable Bokeh backend
hv.extension("bokeh")

## 1. Load Experiment Results

In [ ]:
# Load results from experiments directory
results_dir = Path("../experiments/results")
df = pd.DataFrame()  # Initialize empty dataframe

if not results_dir.exists():
    print(f"❌ Results directory not found: {results_dir}")
    print("\nPlease run experiments first:")
    print("  python experiments/scripts/run_all.py --seeds 42 43 --quick")
else:
    df = load_experiment_results(results_dir)

    if df.empty:
        print("❌ No experiment results found.")
        print("\nPlease run experiments first:")
        print("  python experiments/scripts/run_all.py --seeds 42 43 --quick")
    else:
        print(f"✅ Loaded {len(df):,} data points")
        print(f"\nMethods: {sorted(df['method'].unique())}")
        print(f"Seeds: {sorted(df['seed'].unique())}")
        print(f"Epochs: {df['epoch'].min()} to {df['epoch'].max()}")

        # Show first few rows
        print("\nSample data:")
        display(df.head(10))

## 2. Summary Statistics

In [16]:
# Print summary statistics
if not df.empty:
    print_summary_statistics(results_dir)


SUMMARY STATISTICS

PN:
  Seeds: [np.int64(42), np.int64(43)]
  Test Error: 0.5074 ± 0.0000
  Test Accuracy: 0.4926 ± 0.0000
  Test Loss: 0.0000 ± 0.0000

nnPU:
  Seeds: [np.int64(42), np.int64(43)]
  Test Error: 0.3432 ± 0.2112
  Test Accuracy: 0.6568 ± 0.2112
  Test Loss: 0.8182 ± 0.0267

uPU:
  Seeds: [np.int64(42), np.int64(43)]
  Test Error: 0.3264 ± 0.0922
  Test Accuracy: 0.6736 ± 0.0922
  Test Loss: 1.1623 ± 0.5309




## 3. Interactive Training Curves

These plots show the mean ± standard deviation across all random seeds.
- **Hover** to see exact values
- **Pan and zoom** to explore specific regions
- **Click legend** to show/hide methods

### 3.1 Training Loss

In [ ]:
train_loss_plot = None
if not df.empty:
    train_loss_plot = plot_training_curves(
        df,
        metric="train_loss",
        title="Training Loss",
        width=900,
        height=500,
    )
train_loss_plot

### 3.2 Test Loss

In [ ]:
test_loss_plot = None
if not df.empty:
    test_loss_plot = plot_training_curves(
        df,
        metric="test_loss",
        title="Test Loss",
        width=900,
        height=500,
    )
test_loss_plot

### 3.3 Test Error (Zero-One Loss)

**Key metric**: Lower is better. This is the primary evaluation metric from the nnPU paper.

In [ ]:
test_error_plot = None
if not df.empty:
    test_error_plot = plot_training_curves(
        df,
        metric="test_zero_one_loss",
        title="Test Error (Zero-One Loss)",
        width=900,
        height=500,
    )
test_error_plot

### 3.4 Test Accuracy

In [ ]:
test_acc_plot = None
if not df.empty:
    test_acc_plot = plot_training_curves(
        df,
        metric="test_accuracy",
        title="Test Accuracy",
        width=900,
        height=500,
    )
test_acc_plot

## 4. Final Performance Comparison

In [ ]:
final_perf_plot = None
if not df.empty:
    final_perf_plot = plot_final_performance(
        df,
        width=700,
        height=500,
    )
final_perf_plot

## 5. Complete Comparison Dashboard

All metrics in a single view:

In [ ]:
dashboard = None
if not df.empty:
    dashboard = create_comparison_dashboard(results_dir)
dashboard

## 6. Custom Analysis

### 6.1 Per-Seed Performance

In [23]:
if not df.empty:
    # Get final epoch results for each seed
    final_results = df.loc[df.groupby(["method", "seed"])["epoch"].idxmax()]

    # Pivot table for easy comparison
    pivot_error = final_results.pivot(index="seed", columns="method", values="test_zero_one_loss")
    pivot_acc = final_results.pivot(index="seed", columns="method", values="test_accuracy")

    print("Final Test Error Rate (Zero-One Loss):")
    display(pivot_error.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r", axis=None))

    print("\nFinal Test Accuracy:")
    display(pivot_acc.style.format("{:.4f}").background_gradient(cmap="RdYlGn", axis=None))

Final Test Error Rate (Zero-One Loss):


ImportError: `Import matplotlib` failed. Styler.background_gradient requires matplotlib. Use pip or conda to install the matplotlib package.


Final Test Accuracy:


ImportError: `Import matplotlib` failed. Styler.background_gradient requires matplotlib. Use pip or conda to install the matplotlib package.

### 6.2 Learning Speed Comparison

Which method reaches target performance fastest?

In [24]:
if not df.empty:
    # Target: 95% accuracy
    target_acc = 0.95

    print(f"Epochs to reach {target_acc:.1%} accuracy:\n")

    for method in sorted(df["method"].unique()):
        method_data = df[df["method"] == method]
        epochs_to_target = []

        for seed in sorted(method_data["seed"].unique()):
            seed_data = method_data[method_data["seed"] == seed]
            reached = seed_data[seed_data["test_accuracy"] >= target_acc]

            if not reached.empty:
                first_epoch = reached["epoch"].min()
                epochs_to_target.append(first_epoch)

        if epochs_to_target:
            mean_epochs = sum(epochs_to_target) / len(epochs_to_target)
            print(f"{method:6s}: {mean_epochs:5.1f} epochs (avg across {len(epochs_to_target)} seeds)")
        else:
            print(f"{method:6s}: Did not reach target")

Epochs to reach 95.0% accuracy:

PN    : Did not reach target
nnPU  : Did not reach target
uPU   : Did not reach target


### 6.3 Method Comparison Statistics

In [25]:
if not df.empty:
    # Final epoch statistics
    final_results = df.loc[df.groupby(["method", "seed"])["epoch"].idxmax()]

    # Group by method and compute statistics
    summary = final_results.groupby("method").agg({
        "test_zero_one_loss": ["mean", "std", "min", "max"],
        "test_accuracy": ["mean", "std", "min", "max"],
        "train_loss": ["mean", "std"],
        "test_loss": ["mean", "std"],
    }).round(4)

    print("Method Comparison (Final Epoch):")
    display(summary)

Method Comparison (Final Epoch):


test_zero_one_loss                         test_accuracy          \
                     mean     std     min     max          mean     std   
method                                                                    
PN                 0.5074  0.0000  0.5074  0.5074        0.4926  0.0000   
nnPU               0.3432  0.2112  0.1939  0.4926        0.6568  0.2112   
uPU                0.3264  0.0922  0.2612  0.3916        0.6736  0.0922   

                       train_loss         test_loss          
           min     max       mean     std      mean     std  
method                                                       
PN      0.4926  0.4926     0.0000  0.0000    0.0000  0.0000  
nnPU    0.5074  0.8061     0.3411  0.0304    0.8182  0.0267  
uPU     0.6084  0.7388     0.2992  0.0522    1.1623  0.5309

## 7. Export Results

Export all plots to HTML files for sharing:

In [26]:
if not df.empty:
    from pu_learning.utils.visualization import export_plots

    output_dir = Path("../experiments/results/plots")
    export_plots(results_dir, output_dir)

    print(f"\n✅ Plots exported to: {output_dir}")
    print("\nOpen comparison_dashboard.html in your browser for interactive exploration!")

Saved: ../experiments/results/plots/training_loss.html
Saved: ../experiments/results/plots/test_loss.html
Saved: ../experiments/results/plots/test_error.html
Saved: ../experiments/results/plots/test_accuracy.html
Saved: ../experiments/results/plots/final_performance.html
Saved: ../experiments/results/plots/comparison_dashboard.html

✅ Plots exported to: ../experiments/results/plots

Open comparison_dashboard.html in your browser for interactive exploration!


## Summary

This notebook provides comprehensive analysis of PU learning experiments:

1. **Training curves**: Visualize learning progress over epochs
2. **Performance comparison**: Compare final results across methods
3. **Per-seed analysis**: Examine variance across random seeds
4. **Learning speed**: Identify which method converges fastest
5. **Export capabilities**: Save interactive plots as standalone HTML files

### Key Findings (Example - depends on your results):

- **nnPU** should achieve the best final performance with lowest variance
- **uPU** may show more variance due to potentially negative risk
- **PN** baseline provides strong performance with full supervision

### Next Steps:

1. Run full experiments with more seeds for statistical significance
2. Compare results with nnPU paper benchmarks
3. Experiment with different hyperparameters
4. Try different positive class configurations